# 08 --  Final Model Export

## Concept
The final step: train the best model, evaluate thoroughly, save to disk, and prepare for production deployment.

## Mathematical Intuition
This notebook synthesizes everything from 01-07 into a complete ML pipeline:
1. Load & preprocess data
2. Feature engineering
3. Train best model (hopefully XGBoost or LightGBM)
4. Evaluate on all metrics
5. Save model + scaler + metadata
6. Log to MLflow

## Interview Questions
1. What information should be saved alongside a trained model?
2. How do you ensure reproducibility of model training?
3. What's the deployment strategy for a production model?

## Production Mapping
Models are registered in `ml.model_versions` via ModelRegistry. The ModelPredictor in `inference/predictor.py` loads the latest production model.


In [ ]:
import pandas as pd, numpy as np, json, joblib, os
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
n = 2000
X = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'region_risk': np.random.uniform(0, 1, n),
    'age_years': np.random.exponential(30, n),
    'num_connections': np.random.poisson(5, n),
})
y = (X['capacity_mw'] / 100 + X['region_risk'] * 5 + np.random.normal(0, 0.5, n)).clip(0, 3).round().astype(int).clip(0, 3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Train final model (XGBoost --  best baseline)
model = XGBClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=0
)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
cv_scores = cross_val_score(model, X_train_s, y_train, cv=5)
print(f"Test Accuracy: {acc:.4f}")
print(f"CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix --  XGBoost')
plt.tight_layout()
plt.savefig('../artifacts/08_confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
# Save model, scaler, and metadata
model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)
joblib.dump(model, model_dir / 'energy_criticality_model.joblib')
joblib.dump(scaler, model_dir / 'scaler.joblib')
metadata = {
    'model_type': 'XGBClassifier',
    'features': list(X.columns),
    'n_classes': int(len(model.classes_)),
    'test_accuracy': float(acc),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'params': model.get_params(),
    'random_seed': 42,
}
with open(model_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Model saved to {model_dir.resolve()}")
print(json.dumps(metadata, indent=2))

In [ ]:
# MLflow logging (if mlflow is available)
import mlflow
try:
    mlflow.set_tracking_uri('file:../mlruns')
    mlflow.set_experiment('energy_criticality_final')
    with mlflow.start_run(run_name='final_xgboost'):
        mlflow.log_params(model.get_params())
        mlflow.log_metric('test_accuracy', acc)
        mlflow.log_metric('cv_mean', cv_scores.mean())
        mlflow.log_artifact(str(model_dir / 'energy_criticality_model.joblib'))
        mlflow.log_artifact(str(model_dir / 'scaler.joblib'))
        mlflow.log_artifact(str(model_dir / 'model_metadata.json'))
        print(f"MLflow run logged: {mlflow.active_run().info.run_id}")
    mlflow.end_run()
except Exception as e:
    print(f"MLflow logging skipped: {e}")

In [ ]:
# Load and verify
loaded = joblib.load(model_dir / 'energy_criticality_model.joblib')
loaded_scaler = joblib.load(model_dir / 'scaler.joblib')
sample = X_test[:5]
sample_s = loaded_scaler.transform(sample)
preds = loaded.predict(sample_s)
probs = loaded.predict_proba(sample_s)
for i, (idx, row) in enumerate(sample.iterrows()):
    print(f"Sample {i+1}: true={y_test.iloc[i]}, pred={preds[i]}, confidence={probs[i].max():.4f}")

## Summary --  End-to-End ML Pipeline

### What we built
1. **EDA** --  understanding data distributions and relationships
2. **Preprocessing** --  scaling, encoding, imputing
3. **Feature Engineering** --  ratios, distances, aggregates
4. **Baseline Models** --  LogReg, Decision Tree, Random Forest
5. **Model Comparison** --  XGBoost and LightGBM outperform baselines
6. **Hyperparameter Tuning** --  Grid Search, Random Search, Optuna
7. **Explainability** --  SHAP, permutation importance, feature importance
8. **Export** --  trained model saved with metadata for production

### Production Path
```
research/models/model.joblib + scaler.joblib
    -> services/ml-platform/models/
    -> registered in ml.model_versions
    -> served by ModelPredictor at POST /api/v1/ml/predict
```

### Architecture Principles
- Research = experimentation (notebooks, visualizations, exploration)
- Production = deterministic (versioned features, known params, registry)
- All training runs logged to MLflow with full parameter/metric/artifact tracking
- Every prediction includes model_version, feature_version, confidence, probabilities, latency
